In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse

In [ ]:
df = pd.read_csv("C:\\Personal\\Masters\\Masters_work\\Study\\Y2_S1\\PRT661\\Project\\Data-Science-practise-PRT661\\Machine_Learning\\Attempt_2\\LTHC_phase2_cleaned.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (16184, 12)


,Country of birth of person,Years spent in Australia,Age group,Sex,Long-term health condition (LTHC),Number of people reporting LTHC(s),Population,Age-specific percentage of population reporting LTHC(s),Language used at home,Proficiency in spoken English,Region_class,Subregion_class
0,New Zealand,0–10 years,00–44,Persons,Arthritis,1005.0,100403.0,1.000966,Maori (New Zealand),Not specified,Oceania and Antarctica,New Zealand
1,New Zealand,0–10 years,00–44,Male,Arthritis,453.0,50915.0,0.889718,Maori (New Zealand),Not specified,Oceania and Antarctica,New Zealand
2,New Zealand,0–10 years,00–44,Female,Arthritis,552.0,49488.0,1.115422,Maori (New Zealand),Not specified,Oceania and Antarctica,New Zealand
3,New Zealand,0–10 years,45–64,Persons,Arthritis,1323.0,17280.0,7.656250,Maori (New Zealand),Not specified,Oceania and Antarctica,New Zealand
4,New Zealand,0–10 years,45–64,Male,Arthritis,427.0,8301.0,5.143959,Maori (New Zealand),Not specified,Oceania and Antarctica,New Zealand


In [ ]:
target = "Age-specific percentage of population reporting LTHC(s)"

print("Target:", target)
print("\nTarget statistics:")
print(df[target].describe())

Target: Age-specific percentage of population reporting LTHC(s)

Target statistics:
count    16184.000000
mean        12.223876
std         15.344985
min          0.005603
25%          2.457033
50%          6.728442
75%         15.374186
max         90.000000
Name: Age-specific percentage of population reporting LTHC(s), dtype: float64


In [ ]:
features = [
    "Country of birth of person",
    "Years spent in Australia",
    "Age group",
    "Sex",
    "Language used at home",
    "Proficiency in spoken English",
    "Region_class",
    "Subregion_class",
    "Long-term health condition (LTHC)" 
]

X = df[features]
y = df[target]

print("Features:")
for feature in features:
    print("-", feature)

print("\nTarget:")
print(target)

Features:
- Country of birth of person
- Years spent in Australia
- Age group
- Sex
- Language used at home
- Proficiency in spoken English
- Region_class
- Subregion_class
- Long-term health condition (LTHC)

Target:
Age-specific percentage of population reporting LTHC(s)


In [ ]:
leakage_columns = [
    "Number of people reporting LTHC(s)",
    "Population"
]

print("Potential leakage columns excluded:")
for col in leakage_columns:
    print("-", col)

print("\nFeatures being used:")
print(X.columns.tolist())

Potential leakage columns excluded:
- Number of people reporting LTHC(s)
- Population

Features being used:
['Country of birth of person', 'Years spent in Australia', 'Age group', 'Sex', 'Language used at home', 'Proficiency in spoken English', 'Region_class', 'Subregion_class', 'Long-term health condition (LTHC)']


In [ ]:
group_columns = [
    "Country of birth of person",
    "Years spent in Australia",
    "Age group",
    "Sex",
    "Language used at home",
    "Proficiency in spoken English",
    "Region_class",
    "Subregion_class"
]

df["demographic_group"] = (
    df[group_columns]
    .astype(str)
    .agg("|".join, axis=1)
)

print(
    "Number of demographic groups:",
    df["demographic_group"].nunique()
)

Number of demographic groups: 2717


In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=df["demographic_group"]
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 13060
Testing samples: 3124


In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=df["demographic_group"]
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 13060
Testing samples: 3124


In [ ]:
train_groups = set(
    df.iloc[train_idx]["demographic_group"]
)

test_groups = set(
    df.iloc[test_idx]["demographic_group"]
)

overlap = train_groups.intersection(test_groups)

print("Training groups:", len(train_groups))
print("Testing groups:", len(test_groups))
print("Overlapping groups:", len(overlap))

Training groups: 2173
Testing groups: 544
Overlapping groups: 0


In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical features:")

for col in categorical_features:
    print("-", col)

Categorical features:
- Country of birth of person
- Years spent in Australia
- Age group
- Sex
- Language used at home
- Proficiency in spoken English
- Region_class
- Subregion_class
- Long-term health condition (LTHC)


<ipython-input-10-206c916c8195>:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ]
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (13060, 402)
Processed testing shape: (3124, 402)


In [ ]:
print("Training target:")
print(y_train.describe())

print("\nTesting target:")
print(y_test.describe())

Training target:
count    13060.000000
mean        12.190565
std         15.372291
min          0.005603
25%          2.448379
50%          6.696529
75%         15.247046
max         90.000000
Name: Age-specific percentage of population reporting LTHC(s), dtype: float64

Testing target:
count    3124.000000
mean       12.363135
std        15.231963
min         0.012530
25%         2.505523
50%         6.908001
75%        15.792801
max        88.679245
Name: Age-specific percentage of population reporting LTHC(s), dtype: float64
